# Laboratorium 4: Segmentacja obrazów — filtr Frangiego i sieć U-Net

## Inteligentne przetwarzanie sygnałów

*(instrukcja Prof. Hachaja: „lab_03 — segmentacja”; poniżej jej treść, dalej zasady obowiązujące na tych ćwiczeniach)*

- Maksymalna liczba punktów: 10
- Skala ocen za punkty: 9–10 ~ bardzo dobry (5.0); 8 ~ plus dobry (4.5); 7 ~ dobry (4.0); 6 ~ plus dostateczny (3.5); 5 ~ dostateczny (3.0); 0–4 ~ niedostateczny (2.0)

Na tym laboratorium dokonamy segmentacji obrazów przy pomocy sieci U-Net. Użyjemy zbioru danych „retina”.

1. **[4 punkty]** Dokonaj segmentacji zbioru danych „retina” przy użyciu filtra Frangiego. Określ jej skuteczność przy użyciu krzywej ROC oraz współczynnika ROC AUC na zbiorze testowym.
2. **[4 punkty]** Wytrenuj sieć o architekturze U-Net, która dokona segmentacji zbioru danych „retina”, wykorzystując zbiór treningowy. Możesz wytrenować sieć od zera (bez użycia pretrenowanych wag sieci backbone) lub użyć wersji pretrenowanej. Następnie sprawdź skuteczność segmentacji przy użyciu krzywej ROC oraz współczynnika ROC AUC na zbiorze testowym.
3. **[2 punkty]** Zestaw wyniki obu algorytmów, dokonaj ich podsumowania oraz oblicz współczynniki Dice i IoU.

---

## Zasady na tych ćwiczeniach

- **Środowisko**: podstawowo lokalnie, w środowisku z `requirements_python3.10.txt` Prof. Hachaja (TensorFlow 2.10 / Keras 2.10 / scikit-image 0.19 — notatnik jest z nim zgodny, a skrypty z wykładu działają bez zmian poza ścieżkami). Jeśli `import tensorflow` kończy się błędem `TypeError: Unable to convert function return value to a Python type!`, masz numpy 2.x, z którym TensorFlow 2.10 nie działa: `pip install "numpy==1.26.4" "opencv-python<4.11"` (w pliku `requirements` `opencv-python==4.13.0.92` wymaga numpy 2 — te dwie wersje kolidują ze sobą). Opcjonalnie Colab (TensorFlow 2.19+ / Keras 3) — wtedy skrypty z wykładu wymagają kilku poprawek, wypisanych w komórce „Praca w Colabie” poniżej; GPU włącz **zanim** uruchomisz cokolwiek (*Runtime → Change runtime type → T4 GPU*; zmiana środowiska później kasuje zmienne). Bez GPU trening U-Net też się da zrobić, w mniejszej konfiguracji (hint 2) — komórka 0 wypisuje, czy GPU jest.
- **Dane**: zbiór „retina” (20 obrazów treningowych i 20 testowych z ręczną segmentacją naczyń i maską pola widzenia) i wagi U-Net Prof. Hachaja są w zipie wykładu 02, który pobraliście przed zajęciami. Lokalnie: w komórce 0 wpisz ścieżkę do rozpakowanego zipa (`SCIEZKA_WYKLADU`). W Colabie: komórka 0 pobierze zip z dysku AGH (298 MB, zwykle < 1 min).
- **Wysyłka o 85. minucie**: plik `lab4_<nr_indeksu>.ipynb` **z wynikami komórek** (nie czyścić wyjść). Lokalnie: zapisz notatnik. Colab: *File → Download → Download .ipynb* (po restarcie środowiska wyjścia znikają — wtedy *Run all* przed pobraniem).
- **Hinty**: w komórce na końcu notatnika; hasła podam o 20., 40., 55. i 70. minucie. Hint mówi *co* i *czym* zrobić; zawiera co najwyżej drobne fragmenty kodu, nie gotowe rozwiązania punktowanych części.
- **Kod wykładu** (`src/Run_Frangi_test.py`, `src/baseline_aug.py`, `src/baseline_aug_predict.py` z zipa, który pobraliście przed zajęciami) wolno kopiować i adaptować — pod warunkiem, że rozumiesz, co kopiujesz. W środowisku Prof. Hachaja działa po zmianie ścieżek; w Colabie — z poprawkami z komórki „Praca w Colabie”. **Nie uruchamiaj `baseline_aug.py` w katalogu `src/` zipa**: skrypt na starcie wczytuje plik wag o tej samej nazwie co wagi Prof. Hachaja (więc „trening od zera” zaczyna się od gotowej sieci), a na końcu go nadpisuje — skopiuj skrypt do własnego katalogu.
- **Komentarze**: każde zadanie kończy się komórką tekstową z komentarzem do wyników. Rzeczowe komentarze (co wyszło, dlaczego, co by to poprawiło) są ważnym elementem oceny.
- Zadanie 3 wymaga wyników zadań 1 i 2 — jeśli trening U-Net się nie udał, zadanie 3 można zrobić na gotowych wagach (komórka 2d), z komentarzem, że tak zrobiono.


## Praca w Colabie — jak uruchomić skrypty Prof. Hachaja (pomiń, jeśli pracujesz lokalnie w jego środowisku)

Skrypty z wykładu były pisane pod TensorFlow 2.10 / Keras 2.10. W Colabie jest TensorFlow 2.19+ / **Keras 3** i trzeba nanieść kilka
poprawek. Lista jest kompletna i sprawdzona (każdy skrypt uruchomiony po poprawkach pod Keras 3). Komórka 0 wypakowuje skrypty i wagi do `src/`,
a dane do `retina/`, więc poniższe ścieżki są względem katalogu roboczego Colaba.

**Wszystkie skrypty**
- ścieżki Windows (`c:/wykłady/…/data/retina/`, `c:\\data\\retina\\…`) → `retina/…`; w `Run_Frangi_test.py` dodatkowo
  `os.makedirs("results_frangi/retina", exist_ok=True)` przed zapisem i usunięcie pętli po `usg`/`brain` (tych danych nie ma w zipie);
- `aug_utils.py` importuje `keras_preprocessing`, którego w Colabie nie ma: `!pip install keras_preprocessing` (działa z Keras 3) **albo**
  zamiast `gen()`/`aug_utils` użyj `losuj_laty` z komórki 0 (zalecane — prościej i bez instalacji).

**`baseline_aug.py`, `baseline_aug_vgg.py` (trening)**
- `Adam(lr=1e-3)` → `Adam(learning_rate=1e-3)` (z `lr` Keras 3 rzuca `ValueError`);
- `model.fit_generator(gen(...), epochs=…, steps_per_epoch=…, use_multiprocessing=False, workers=1)` →
  `model.fit(gen(...), epochs=…, steps_per_epoch=…)` (bez dwóch ostatnich argumentów); z `losuj_laty`: `model.fit(laty, maski, batch_size=16, epochs=…)`;
- `model.save_weights(file_path)` → nazwa pliku **musi** kończyć się na `.weights.h5`, np. `model.save_weights(model_name + "weights.best.weights.h5")`
  (stare `.hdf5` Keras 3 czyta, ale nie zapisuje); `load_weights(..., by_name=True)` na początku skryptu działa tylko dla plików `.hdf5` Prof. Hachaja —
  własny plik `.weights.h5` wczytuj bez `by_name`;
- `epochs=100`, `steps_per_epoch=125` — to 200 tys. łat; na zajęcia zmniejsz (hint 2);
- `baseline_aug_vgg.py`: `VGG_PATH = "src/vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5"` (plik jest w zipie wykładu; komórka 0 go wypakowała).

**Jak uruchamiać:** z katalogu roboczego, np. `!python src/baseline_aug.py` — wtedy importy `aug_utils` / `from baseline_aug import get_unet` działają, a ścieżki `retina/…` i `src/…` się zgadzają. Kopiując kod do komórek notatnika, najpierw `import sys; sys.path.insert(0, "src")`.

**`baseline_aug_predict.py`, `baseline_aug_vgg_predict.py` (ocena)**
- usuń `from pycocotools import mask as maskUtils` (nieużywane; pakietu nie ma w Colabie); `from skimage.morphology import label` może zostać;
- ścieżka do wag Prof. Hachaja: `file_path = "src/" + model_name + "weights.best.hdf5"`; `load_weights(file_path, by_name=True)` działa wtedy bez zmian
  (sprawdzone: AUC 0,978 jak w jego wynikach). Pętlę po pikselach (`for p … for q …`) można zastąpić indeksowaniem maską: `pred_[mask_[..., 0] > 0.5]`, `gt_[..., 0][mask_[..., 0] > 0.5]`.

**Filtr Frangiego — inna wersja scikit-image.** W Colabie jest scikit-image ≥ 0.20, w środowisku Prof. Hachaja 0.19. Od wersji 0.20 `frangi` jest liczony
**bez normalizacji skali** (mnożenia hesjanu przez σ², opisanego w artykule Frangiego) i z automatycznym `gamma`; te same parametry dają w Colabie
AUC ok. 0,86 zamiast ok. 0,92. Oba wyniki są „poprawne”, ale żeby porównywać się z osobami pracującymi lokalnie, licz każdą skalę osobno:
`np.max([frangi(x, sigmas=[s], gamma=15 / s**2) for s in sigmy], axis=0)` — to odtwarza wersję 0.19 (sprawdzone: ta sama AUC). W komentarzu do zad. 1
napisz, którego wariantu użyłeś/aś.

**Nie dotyczy Colaba, ale warto wiedzieć:** `custom_activation` używa `K.relu`, którego w Keras 3 nie ma — funkcja jest nieużywana, więc nic się
nie dzieje, dopóki jej nie wywołasz. `tensorflow_addons` (sieć Frangi, `src/fn/`) nie instaluje się pod nowy TensorFlow — ta część wykładu
nie jest na zajęciach potrzebna.


In [ ]:
# ============================================================================
# KOMÓRKA 0 — środowisko, dane, funkcje pomocnicze (uruchom raz; w Colabie GPU włącz PRZED tą komórką)
# ============================================================================
import os, glob, zipfile, urllib.request, warnings, time
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
try:                                                            # wycisza komunikaty startowe TensorFlow (C++ pisze na stderr)
    _fd2 = os.dup(2); _nul = os.open(os.devnull, os.O_WRONLY); os.dup2(_nul, 2); os.close(_nul)
except OSError:
    _fd2 = None
import tensorflow as tf
import keras
from keras import layers
if _fd2 is not None: os.dup2(_fd2, 2); os.close(_fd2)
import skimage
from skimage.filters import frangi
from sklearn.metrics import roc_curve, roc_auc_score

print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| scikit-image", skimage.__version__)
_gpu = tf.config.list_physical_devices("GPU")
print("GPU:", _gpu[0].name if _gpu else "BRAK — trening U-Net będzie wolny (patrz zasady i hint 2)")

# --- dane ---------------------------------------------------------------------
# Zbiór "retina" i wagi U-Net są w zipie wykładu 02 ("Wykład 02 - Segmentacja.zip"), który pobraliście przed zajęciami.
# PRACA LOKALNA: wpisz niżej ścieżkę do ROZPAKOWANEGO zipa (katalog, w którym są podkatalogi data/, src/, img/) — nic nie jest kopiowane.
# COLAB: zostaw pustą — komórka pobierze zip z udziału Prof. Hachaja na dysku AGH (298 MB, zwykle < 1 min) i wyciągnie potrzebne pliki
# (data/retina -> retina/, src/*.py i wagi -> src/).
# Awaryjnie (bez sieci do dysku AGH): wgraj do katalogu roboczego retina.zip albo cały zip wykładu pod oryginalną nazwą.
SCIEZKA_WYKLADU = r""        # np. r"C:\studia\IPS\Wykład 02 - Segmentacja"  albo  "/home/ja/ips/Wykład 02 - Segmentacja"
URL_ZIP_WYKLADU = "https://dysk.agh.edu.pl/s/YaYYYepRrHritMo/download?path=/&files=Wyk%C5%82ad%2002%20-%20Segmentacja.zip"
PLIKI_WAG = ["baseline_unet_aug_retina_0.1_activation_ReLU_weights.best.hdf5", "baseline_unet_aug_vgg_retina_0.1_activation_ReLU_weights.best.hdf5"]   # w zipie w katalogu src/

def _wyciagnij_z_zipa_wykladu(sciezka_zip):
    with zipfile.ZipFile(sciezka_zip) as z:
        for m in z.namelist():
            if m.startswith("data/retina/") and not m.endswith("/"):
                cel = m[len("data/"):]
                os.makedirs(os.path.dirname(cel), exist_ok=True)
                with z.open(m) as src, open(cel, "wb") as dst: dst.write(src.read())
            elif m.startswith("src/") and m.count("/") == 1 and m.endswith((".py", ".h5", ".hdf5")):
                os.makedirs("src", exist_ok=True)                 # skrypty wykładu + wagi (U-Net, U-Net+VGG16, VGG16-ImageNet)
                if not os.path.exists(m):
                    with z.open(m) as src, open(m, "wb") as dst: dst.write(src.read())

if SCIEZKA_WYKLADU:                                            # lokalnie: używamy plików w miejscu
    KATALOG = os.path.join(SCIEZKA_WYKLADU, "data", "retina"); KATALOG_SRC = os.path.join(SCIEZKA_WYKLADU, "src")
    assert os.path.isdir(KATALOG), f"nie ma katalogu {KATALOG} — sprawdź SCIEZKA_WYKLADU"
else:                                                          # Colab / katalog roboczy
    KATALOG = "retina"; KATALOG_SRC = "src"
    if not os.path.isdir(KATALOG):
        if os.path.exists("retina.zip"):
            with zipfile.ZipFile("retina.zip") as z: z.extractall(".")
        else:
            lokalne = [p for p in ("Wykład 02 - Segmentacja.zip", "wyklad02.zip") if os.path.exists(p)]
            if not lokalne:
                print("Pobieram zip wykładu z dysku AGH (298 MB) ...", flush=True)
                t0 = time.time(); urllib.request.urlretrieve(URL_ZIP_WYKLADU, "wyklad02.zip")
                print("pobrano w %.0f s" % (time.time() - t0)); lokalne = ["wyklad02.zip"]
            else:
                print("używam lokalnego pliku:", lokalne[0])
            _wyciagnij_z_zipa_wykladu(lokalne[0])
    elif not all(os.path.exists(os.path.join("src", p)) for p in PLIKI_WAG):
        for p in ("Wykład 02 - Segmentacja.zip", "wyklad02.zip"):
            if os.path.exists(p): _wyciagnij_z_zipa_wykladu(p); break
SCIEZKI_WAG = [os.path.join(KATALOG_SRC, p) for p in PLIKI_WAG]
PLIK_WAG, PLIK_WAG_VGG = SCIEZKI_WAG
assert os.path.isdir(os.path.join(KATALOG, "training", "images")), f"zły układ katalogów w {KATALOG} (oczekiwano training/images)"
print("dane:", os.path.abspath(KATALOG), "| skrypty i wagi z wykładu:", os.path.abspath(KATALOG_SRC))
print("wagi Prof. Hachaja dostępne:", [os.path.basename(p) for p in SCIEZKI_WAG if os.path.exists(p)] or "brak (bez wpływu na zad. 1–2; komórka 2d ich wymaga)")

def wczytaj_zbior(czesc):
    """czesc = 'training' albo 'test'. Zwraca (X, Y, FOV):
       X   uint8 (N, 584, 565, 3) — obrazy RGB dna oka,
       Y   bool  (N, 584, 565)    — ręczna segmentacja naczyń (ground truth),
       FOV bool  (N, 584, 565)    — maska pola widzenia (True wewnątrz koła; poza nim obraz jest czarny)."""
    pliki = lambda wzor: sorted(glob.glob(os.path.join(KATALOG, czesc, wzor)))
    X = np.stack([np.array(Image.open(p).convert("RGB")) for p in pliki("images/*.tif")])
    Y = np.stack([np.array(Image.open(p).convert("L")) > 127 for p in pliki("*_manual/*")])
    FOV = np.stack([np.array(Image.open(p).convert("L")) > 127 for p in pliki("mask/*.gif")])
    assert len(X) == len(Y) == len(FOV) == 20, f"oczekiwano 20 obrazów/masek, jest {len(X)}/{len(Y)}/{len(FOV)} — w {czesc}/ ma być dokładnie jeden katalog *_manual"
    return X, Y, FOV

X_tr, Y_tr, FOV_tr = wczytaj_zbior("training")
X_te, Y_te, FOV_te = wczytaj_zbior("test")
print("training:", X_tr.shape, X_tr.dtype, "| test:", X_te.shape,
      "| piksele naczyń w polu widzenia: %.1f %%" % (100 * Y_te[FOV_te].mean()))

# --- funkcje pomocnicze -------------------------------------------------------
def pokaz(obrazy, tytuly=None, kolumny=4, rozmiar=4):
    """Siatka obrazów; obrazy 2-D (szarość, maski, odpowiedzi filtrów) rysowane w skali szarości."""
    n = len(obrazy); wiersze = -(-n // kolumny)
    fig, osie = plt.subplots(wiersze, kolumny, figsize=(rozmiar * kolumny, rozmiar * wiersze * 584 / 565))
    for k, ax in enumerate(np.atleast_1d(osie).ravel()):
        ax.axis("off")
        if k < n:
            ax.imshow(obrazy[k], cmap="gray" if np.ndim(obrazy[k]) == 2 else None)
            if tytuly: ax.set_title(tytuly[k], fontsize=10)
    plt.tight_layout(); plt.show()

def losuj_laty(X, Y, ile, rozmiar=64, rng=None):
    """`ile` losowych wycinków rozmiar×rozmiar z obrazów X (uint8) i masek Y (bool), z losowymi odbiciami
       (uproszczony odpowiednik generatora z baseline_aug.py). Zwraca:
       laty  float32 (ile, rozmiar, rozmiar, 3) w [0, 1],   maski float32 (ile, rozmiar, rozmiar, 1) w {0, 1}."""
    rng = rng if rng is not None else np.random.default_rng(0)
    N, H, W, _ = X.shape
    laty = np.empty((ile, rozmiar, rozmiar, 3), np.float32)
    maski = np.empty((ile, rozmiar, rozmiar, 1), np.float32)
    for k in range(ile):
        i = rng.integers(N); y = rng.integers(H - rozmiar + 1); x = rng.integers(W - rozmiar + 1)
        a = X[i, y:y + rozmiar, x:x + rozmiar] / 255.0
        b = Y[i, y:y + rozmiar, x:x + rozmiar, None].astype(np.float32)
        if rng.random() < 0.5: a, b = a[:, ::-1], b[:, ::-1]
        if rng.random() < 0.5: a, b = a[::-1], b[::-1]
        laty[k], maski[k] = a, b
    return laty, maski

def dopelnij16(obraz):
    """Dopełnia obraz (H, W[, C]) odbiciem lustrzanym do wymiarów podzielnych przez 16
       (U-Net z czterema poolingami wymaga tego przy predykcji na całym obrazie); wynik przyciąć do (H, W)."""
    H, W = obraz.shape[:2]; H2, W2 = -(-H // 16) * 16, -(-W // 16) * 16
    return np.pad(obraz, ((0, H2 - H), (0, W2 - W)) + ((0, 0),) * (obraz.ndim - 2), mode="reflect")

print("Gotowe. Funkcje: wczytaj_zbior, pokaz, losuj_laty, dopelnij16. Zmienne: KATALOG (dane), KATALOG_SRC (skrypty i wagi), PLIK_WAG.")


In [ ]:
# Wpisz swój numer indeksu — steruje ziarnem losowości i wyborem obrazu do ilustracji.
NR_INDEKSU = 0

assert NR_INDEKSU > 0, "wpisz numer indeksu"
ZIARNO = NR_INDEKSU
MOJ_OBRAZ = NR_INDEKSU % 20          # numer obrazu testowego (0–19) używanego w ilustracjach
rng = np.random.default_rng(ZIARNO)
keras.utils.set_random_seed(ZIARNO)
print("ziarno:", ZIARNO, "| mój obraz testowy:", MOJ_OBRAZ)
pokaz([X_te[MOJ_OBRAZ], Y_te[MOJ_OBRAZ], FOV_te[MOJ_OBRAZ]],
      [f"X_te[{MOJ_OBRAZ}] — obraz", "Y_te — naczynia (ground truth)", "FOV_te — pole widzenia"], kolumny=3)


## Zadanie 1 [4 pkt] — segmentacja filtrem Frangiego, krzywa ROC i AUC

Filtr Frangiego (`skimage.filters.frangi`) nie uczy się niczego: dla każdego piksela liczy z drugich pochodnych obrazu
(hesjanu) „naczyniowość” (*vesselness*) — liczbę tym większą, im bardziej otoczenie piksela wygląda jak fragment ciemnej/jasnej rurki.
Wynik jest ciągły, więc można go oceniać krzywą ROC bez wybierania progu.

**1a.** Dla obrazu `X_te[MOJ_OBRAZ]` policz odpowiedź filtru Frangiego i pokaż obok siebie: obraz, segmentację ręczną `Y_te[MOJ_OBRAZ]`
i odpowiedź filtru. Zdecyduj (i napisz w komentarzu): który kanał/konwersję podajesz na wejście, jakie skale `sigmas`
i jak ustawić parametr decydujący o tym, czy szukamy struktur ciemniejszych, czy jaśniejszych od tła.

**1b.** Policz odpowiedź filtru dla **wszystkich 20 obrazów testowych**. Z pikseli **wewnątrz pola widzenia** (`FOV_te`)
wyznacz krzywą ROC (`roc_curve`) i AUC (`roc_auc_score`), zbiorczo dla całego zbioru testowego. Narysuj krzywą ROC
(AUC w tytule, przekątna jako odniesienie).

**Komentarz do zad. 1** (komórka tekstowa): jakie parametry wybrano i dlaczego; co w obrazie filtr „łapie” poza naczyniami;
jak zachowuje się rozkład wartości odpowiedzi (podpowiedź: wypisz kilka kwantyli). Wersję scikit-image wypisuje komórka 0 — od niej zależy wynik.


In [ ]:
# --- 1a: filtr Frangiego na jednym obrazie -------------------------------------
# TODO: wybierz kanał/konwersję (obraz RGB uint8 -> tablica 2-D float w [0,1]),
#       ustaw sigmas i parametr polaryzacji (ciemne vs jasne struktury), policz `frangi(...)`,
#       pokaż: obraz, Y_te[MOJ_OBRAZ], odpowiedź filtru (odpowiedź ma mały zakres — do wyświetlenia
#       warto ją przeskalować, ale do dalszych obliczeń używaj wartości surowych).



In [ ]:
# --- 1b: cały zbiór testowy, krzywa ROC i AUC --------------------------------
# TODO: odpowiedź filtru dla 20 obrazów testowych -> odp_te (20, 584, 565);
#       piksele w FOV jako wektory: y_te = Y_te[FOV_te] (bool), p_frangi = odp_te[FOV_te] (float);
#       roc_curve + roc_auc_score -> auc_frangi; wykres ROC. (Tych nazw używają kolejne komórki i hinty.)



**Komentarz do zadania 1:** *(tu wpisz: wybór kanału, skal i polaryzacji; co filtr wykrywa poza naczyniami; rozkład odpowiedzi)*

## Zadanie 2 [4 pkt] — U-Net: trening na zbiorze treningowym, ROC i AUC na testowym

Sieć uczymy na **losowych wycinkach (łatach, ang. *patches*) 64×64** z 20 obrazów treningowych
(tak robi `baseline_aug.py` z wykładu — jego generator `gen()` zastępuje tu gotowa funkcja `losuj_laty`), a stosujemy
na **całych obrazach** — sieć jest w pełni splotowa, więc rozmiar wejścia może być inny niż w treningu
(musi być podzielny przez 16: funkcja `dopelnij16`).

**2a.** Zbuduj U-Net dla wejścia `(None, None, 3)` z jednym kanałem wyjściowym i aktywacją sigmoid. Możesz zaadaptować
`get_unet` z `baseline_aug.py` (w Colabie — z poprawkami z komórki „Praca w Colabie”) albo napisać własny, mniejszy. Wypisz `summary()`
i liczbę parametrów.

**2b.** Wylosuj łaty treningowe (`losuj_laty(X_tr, Y_tr, ile, 64, rng)`), skompiluj (Adam, binarna entropia krzyżowa)
i wytrenuj sieć. Liczbę łat i epok dobierz do dostępnych zasobów (GPU/CPU) — napisz w komentarzu, co wybrano i ile trwał trening.
Narysuj przebieg funkcji straty.

**2c.** Policz predykcję dla 20 obrazów testowych (cały obraz naraz, po `dopelnij16`, wynik przycięty do 584×565),
pokaż obraz `MOJ_OBRAZ` z predykcją, a następnie — tak samo jak w 1b — krzywą ROC i AUC z pikseli w polu widzenia.
Obie krzywe (Frangi i U-Net) narysuj na jednym wykresie.

*(Wariant „użyj wersji pretrenowanej” z instrukcji — U-Net z enkoderem VGG16 jak w `baseline_aug_vgg.py` — jest dozwolony, ale ma 18,5 mln parametrów (ponad 2× więcej niż U-Net z wykładu), więc bez GPU nie zmieści się w czasie zajęć.)*

**2d (awaryjnie / do porównania).** Jeśli trening się nie udał lub nie ma GPU: komórka poniżej buduje U-Net o architekturze
z wykładu, wczytuje gotowe wagi Prof. Hachaja i liczy predykcję oraz AUC na teście (zmienne `pred_te`, `p_unet`, `auc_unet` — te same, których
używa zad. 3; ścieżkę do pliku wag ustawiła komórka 0). W komentarzu napisz wyraźnie, że to wagi Prof. Hachaja, nie własny trening.

**Komentarz do zad. 2**: konfiguracja (architektura, liczba parametrów, łat, epok, czas), przebieg straty (czy się uczy, czy
dotrenowanie coś by dało), porównanie AUC z filtrem Frangiego.


In [ ]:
# --- 2a: architektura U-Net ---------------------------------------------------
# TODO: funkcja zbuduj_unet(...) -> keras.Model; wejście layers.Input((None, None, 3)), wyjście Conv2D(1, 1, activation="sigmoid").
#       Enkoder: bloki [Conv2D 3x3 + ReLU] x2 + MaxPooling2D; bottleneck; dekoder: Conv2DTranspose(…, 2, strides=2) + concatenate ze
#       skip connection + [Conv2D 3x3 + ReLU] x2.  model = zbuduj_unet(...); model.summary(); liczba parametrów.



In [ ]:
# --- 2b: łaty treningowe i trening --------------------------------------------
# TODO: laty, maski = losuj_laty(X_tr, Y_tr, ILE, 64, rng); model.compile(Adam, "binary_crossentropy");
#       hist = model.fit(...); wykres straty. Zmierz czas treningu (time.time()).



In [ ]:
# --- 2c: predykcja na zbiorze testowym, ROC i AUC ------------------------------
# TODO: pred_te (20, 584, 565) z model.predict na dopelnij16(obraz/255.) i przycięciem [:584, :565]; pokaż MOJ_OBRAZ;
#       p_unet = pred_te[FOV_te]; ROC + AUC (auc_unet) z tych samych y_te co w 1b; obie krzywe na jednym wykresie.



In [ ]:
# --- 2d (awaryjnie): gotowe wagi U-Net Prof. Hachaja ------------------------------
# Samowystarczalna komórka: architektura DOKŁADNIE jak get_unet w baseline_aug.py (baza 32, Dropout 0.1 po każdym splocie,
# Conv2DTranspose) — inaczej wagi się nie wczytają. Ścieżkę PLIK_WAG ustawiła komórka 0 (lokalnie: src/ rozpakowanego zipa; Colab: src/).
# Uruchom tylko, gdy własny trening się nie udał (albo do porównania z własnym modelem — wtedy nie nadpisuj pred_te).
UZYWAM_WAG_WYKLADU = False          # <- ustaw True, żeby ta komórka zdefiniowała pred_te / p_unet / auc_unet dla zad. 3
if UZYWAM_WAG_WYKLADU:
    assert os.path.exists(PLIK_WAG), "brak pliku wag — sprawdź SCIEZKA_WYKLADU / uruchom ponownie komórkę 0"
    def unet_wykladu(do=0.1):
        inp = layers.Input((None, None, 3)); x = inp; skoki = []
        def blok(x, n):
            x = layers.Dropout(do)(layers.ReLU()(layers.Conv2D(n, 3, padding="same")(x)))
            return layers.Dropout(do)(layers.ReLU()(layers.Conv2D(n, 3, padding="same")(x)))
        for n in (32, 64, 128, 256):
            x = blok(x, n); skoki.append(x); x = layers.MaxPooling2D(2)(x)
        x = blok(x, 512)
        for n, sk in zip((256, 128, 64, 32), reversed(skoki)):
            x = blok(layers.concatenate([layers.Conv2DTranspose(n, 2, strides=2, padding="same")(x), sk]), n)
        return keras.Model(inp, layers.Conv2D(1, 1, activation="sigmoid")(x), name="unet_wykladu")
    model_wyklad = unet_wykladu()
    model_wyklad.load_weights(PLIK_WAG)
    pred_te = np.stack([model_wyklad.predict(dopelnij16(im / 255.0)[None], verbose=0)[0, :584, :565, 0] for im in X_te])
    y_te = Y_te[FOV_te]; p_unet = pred_te[FOV_te]; auc_unet = roc_auc_score(y_te, p_unet)
    print("AUC (test, FOV) wagi Prof. Hachaja: %.4f — w zad. 3 użyte będą TE predykcje (napisz to w komentarzu!)" % auc_unet)


**Komentarz do zadania 2:** *(konfiguracja i czas treningu; przebieg straty; AUC vs Frangi; co by poprawiło wynik)*

## Zadanie 3 [2 pkt] — zestawienie: progi, Dice, IoU, podsumowanie

Dice i IoU porównują **maski binarne**, więc trzeba wybrać próg. Dla U-Net naturalny jest 0,5 (wyjście sigmoid to prawdopodobieństwo).
Dla filtru Frangiego odpowiedź nie jest prawdopodobieństwem — próg trzeba **dobrać na zbiorze treningowym** (np. maksymalizując Dice)
i zastosować do testowego. Wybór progu na zbiorze testowym to wyciek informacji z testu do „treningu”.

**3a.** Dobierz próg dla filtru Frangiego na zbiorze treningowym. Policz Dice i IoU (piksele w polu widzenia, zbiorczo dla 20 obrazów
testowych) dla obu metod. Zestaw w tabeli: AUC, próg, Dice, IoU.

**3b.** Pokaż dla `MOJ_OBRAZ` maski obu metod obok ground truth (dobrze działa mapa błędów: TP/FP/FN w kolorach).

**Podsumowanie** (komórka tekstowa, 5–10 zdań): która metoda jest lepsza i o ile; gdzie każda zawodzi (jakie struktury, jakie
miejsca obrazu); czy różnice między metodami w AUC i w Dice/IoU są podobnej wielkości — a jeśli nie, z czego to wynika; co to mówi
o metrykach przy nierównowadze klas.


In [ ]:
# --- 3a: progi, Dice, IoU --------------------------------------------------------
# TODO: funkcja dice_iou(pred_bin, gt) z TP/FP/FN; odpowiedź Frangiego dla X_tr -> próg maksymalizujący Dice na treningu
#       (kandydaci np. kwantyle odpowiedzi); maski binarne na teście; tabela (pandas) z AUC, progiem, Dice, IoU.



In [ ]:
# --- 3b: maski dla MOJ_OBRAZ ----------------------------------------------------
# TODO: ground truth, maska Frangiego (>= prog), maska U-Net (>= 0.5); np. mapa błędów RGB: R = FP, G = TP, B = FN.



**Podsumowanie (zad. 3):** *(która metoda, o ile; gdzie zawodzą; AUC vs Dice; metryki przy nierównowadze klas)*

## Przed wysyłką — lista kontrolna

- [ ] `NR_INDEKSU` wpisany; wszystkie komórki wykonane w tej sesji (wyjścia widoczne).
- [ ] Zad. 1: obraz + GT + odpowiedź filtru; krzywa ROC z AUC dla 20 obrazów testowych (piksele w FOV); komentarz.
- [ ] Zad. 2: `summary()` i liczba parametrów; wykres straty; czas treningu; predykcja dla `MOJ_OBRAZ`; ROC obu metod na jednym wykresie; komentarz.
- [ ] Zad. 3: próg Frangiego dobrany na zbiorze treningowym (i napisane jak); tabela AUC/próg/Dice/IoU; maski/mapa błędów; podsumowanie.
- [ ] Plik: `lab4_<nr_indeksu>.ipynb` z wynikami komórek (Colab: *File → Download → Download .ipynb*; lokalnie: zapisz notatnik).


In [2]:
# ============================================================================
# HINTY — hasła podaje prowadzący w trakcie zajęć. Użycie: odblokuj("haslo")
# ============================================================================
import base64, hashlib, zlib, unicodedata
from IPython.display import Markdown, display

_HINTY = [
 "H00zMOBjaatT/EzXMkqh/N/bJ2C2dKVQONhofs1QeyVJY1AF/PO+1CHj1vlbUBm4RlUyrxTZqLIibwOsdBIFWyjjVFESUYl6rFDP59zYxqiMGEsBxOkCDOrDMhr227PYtbnT3m+Lm25f5lCsFnNrZFlobPMHLgAZZuUhr2e6222WYAo8PBCMYshrpH0Y7hOT1/9kv8L4WOXI6/8SUdSwxA//FgetDM0/3LJUrw4LDm7CdEyZ8v+NWMg971IKf2vgAKKri0tPCDJp11LE+U2TS8Im4eMyLMlhxp6uQHHHmbgK8zgmtoMmHxZ0KG8I/xwORgZT90OJ6xOcBhLgxdGR8dxn4dKdrcPENatHFnz3O8wRJynujGpwARAo7bYGJd57rx9R/gzSSaBbOY9OYbvEIQL0CORkUcZNE7oc1+ahoRbRSfvM+CNxvIa1Vfrt5GALlF47V58EG8SEgxenqdDIPjvQBo5FOxMU0k8X4/dtuGtdqSOMVPLnqs5o+1ZfSUhOklv5TmJboIZWGkwdMoUC6BeuH7+qw/TTlFFv9lmz0DV22Sq8O90DAs3vwpDXOqWULOlUDihZranfCCZelWtIVzHyge0uKoucRgD22kA+ZykeaYYhaEiHP7g+BT3F4nOBu4VJeoT4GcZj1/lJ8MiNr5F7I4Fs6OWd+Zj5sd1Ws85OJ3gUBpkaJmVkwvxGOyXhPfLp7BYQyTGHYl4mG+e+OGAJvsi0SWTX66iwyj6OFjMftsexOLtJXzVDmS5yMPHvxdbLAlT/X2ogYE6+WMxlPfY0SZqdoVEO9qO7wS+ei7eUua6+xpw33NJR7IyHr3+lB+z9bOH/fpWLpIpAGiev0JD9krPiMfEYsfcru8PZEhRjbAkwV6xXo/NMNkz6H07DO/b2VpTzJJuYqletmn68XvN2I1fn796jBFy03WmrhsANqzWSl5+7gcn/SvqjTDQ7duXfjvM4G+Vo01aKwneZh4ufYLAmssQBGNX8Rxc+rhiemmtAPADC2ej7oseNkUiPshHnXSAcLy3zktXA3xsS1yjs54Kz3rf1kbxLTxsWyFmvFOi8ReCk1KzqNIxnBviqzUg8+SP4nhHL+dbjHoSwmYXp9JA4GHFyasGjVcZgn4bvvzB8h6kj4QlaynsibxwT++vvsYcqRN0I3pZKlmj8OZIK6dOSL/t4WMmF9qzYISF6kcRGh43QqW/BhdrWlaxNjNqAauqUo1FvhhJI5O4tD8NvianVER1cUJoxbsny3IKmOihTe4N7XjoCqhlOcraXn7JVyBtkXBTzXd2wH9mBVI10dMEtwUDVkeUkPksr7NvMCsZbwA1A1nI5ke67QzG8LJSSf1tad7zFsyeubJ8eBrZV97Pb2XoYo+DZXKP0/ptv33nT0CSPcJMhKbbt1cD07jbApEgygx8WeLLOSNFRNcQvPmt1uaTXSmCsK6Q3vAcqDGGif5xHwdZ8R80xU8S1rfVOWXGaDoXxXp+UEgNuajKUtdBtosZwqfw672SzpyjzKxiK/z0j9Og1QWYHu4Lv1XO6Bh+g0R6Pcwifxttj8JKnS853QIl/BiHqi0Gx44K8cv54qF2r32J3mvcn1+9viGEwsOU7R6YXeOTHTQADtBCapHEELhcHAFxzCoT7q5emVw0k5BCCkDuS5AbeWoYUmV8J/a/dWru3Y//l0U5MfyeS+24bW+QhljHJBWVxlaX2vkl0a9wls3BreEDQZ5Rd",
 "90fGa3B1C/Aa9hRZjFrWVyQsD3wGvV51UPo+FBd5txQriU3xnHPS5kFuyHz+13Q130GoplUd3PZvZ6ZT4Jer3KB7bUI0+NuV/UghoYNnFnRTfL6GI2qJJj7Y4+agAJ1fZihE8LNpRnpqWOxON4/INF5UviRXWW4xTsl5XjAQtSwmJjwyfGwXvAEI+BCmubmCAKGV8abSBoZqEu4JBFaor+nTIUBDofMbNMXNJ+z7CEsKQmj/wLxGuiVwBm2OmJTO6rwIMjz5Tp6Oj4zycrxc3TkRdgX/EUAOjnjsPJBAEaVRtYRSqM4S9VCGhtSMlOV2gHF0thML4BBDBaZo6iZ+VzBT2x1K+6dEletKG897sUW/p1GTTTtpjQrhXd21KwzGG495EzrP4DilHgmA6HWk+tnkKOarpmDddkPAW/8afiHfLdDhUPuJ3VkLSl4XS28GUsOP/GFZqm0m6LcstiffSNb6BVe9Zhh3+FyFxDmHiyuwnHyu8RthALwMgPpm0bYbSCy9/S30YeEQpGyd4RtEAW2pPzQyCv1wlQgsdndqXSp9AB0uW54lMxr0A/YLuLCZDI3FhwNc93J4FXKayilVCEA2s5g+77qAVdrCRom9ulYn3EwrlG4pxaYFirxM1k8K8G+yZigUULdkPHV868HPUTZlqFF/CDD37OWYxPGDT3tI//MHjzmfvo/nvTihMxHOd3i0316TWq9Cvl0itH/VbmrshqMtx6bsZ6FOtRXDZYcPJHOgia+oAznAx7wBi58eC5Z5WsWTs44Xgtz0OkxpzCZM4Y0eORqb+rtSmrZq2a/ykiCNdZ8j+IVAdXLGDCP+XzY1WnLNi6fbpt7ZlcTtc2iyQRV5lYws3pywBeXSkU9UbEtb9srxzVYMa4WRC0VtRvqAYHlywlWhTa9x9Lss146UyX76W2rOuTpbu6qshmBenzcm1BY9Re0LDkXRrwGspObiJIMVuH5EpAI72/pvFgxTtMxzB9sHDaIJnYlIk84JJEkKBhQfJnOmyYka3qRONngCXz+rcKqn/kofLYtMcJWPJFVgoJZfMk8Az3BgBX5Cr3qaOOa7D2UoJEXkrfV3bs3H8gFyuLEutA7ZC4xwAcnOPVFSSEJyBchyzh2eB+fhTdP+owV+jjovg7fiCcgecb7f/UXeVjIdOspej2supobKsy87QKFNcyVrnF9gC8TxkBAPUYt2nqnD1wOW0xsDOZ4tNvVlrxZRNegshmacYOKZhr98j5ez1oK2zwUSZKlccR6aXKUgM6kOFw8/mMVRLXBK9bgLsek89yDECMMEMn5bWmgaV4c6LXmOT1vhBXqCW8deV+zOug7+Pc6qoIZJm7drrs63uLdVnpB+H4uLdKzRHud+JOWAAtAp0i+8kaN0iSNdaT++qBsG9KAtI+GeFJIEfyuILDGKxS5bnPwoODMc3SKTKNQOzXt1VZ4HZnX6RwxrRy3WglSjMDxL1pR3Kyqdu0jNaVudaMCC8cg7bUKHmVQJZFTX0Vuy5K1faUzkHFFL4oiV2RetZ81+/dcNwjJLHIUz3v+CnwxYFuxo2I9TbnCZRjvSCfnF1DF6k8/uRX1EXONi/kZFQsLqj+RcENdHUJx+u+6cfilUTTce+O+95Sri+bsBjYAHVWrYWYU4VAw4udi80NVVoMZXPbA5liJIvjRK9NIVRo8rFVFImoP2RYeanAwZu2s35S+HrTpZlNhNSOzFsJW8jKUUGcp+PVUSleP4tlQZ3VtNlD2g0iKqqoy2dSEqzeJmPmZ2efjX04pEY+O6YtTWc6D3IpJt6b/0VtRekZImhOM+Y1zBVeA3bBTAhILEeJgvfWaUXUlWz18qY0WmbS0JOpXRSspBd/7C9H1jzbuHG/RO2vGS7nZzZfTMU/14QBURqGM0c0mWnoStVCsGiKwL+oAngl9Ibt04wITQ1AL9KlHjXPtmXXHzFzJkoVLy4JYTqFVp2Xomum0AJYsy8/7vMm42zTvj7s2v6oH9BfVREYtM6RTtdObjxE48qBt4r2m0ZyfB5Kn1R+uu6OM27SHG84/lNJ7LI/Um0uM4LoyUw08h1oFVtz7pKrOhyz5WeWYsvVJJjxRw8WinBSo6PS4cBQahQ1XezPLHpU/ktKv+LSEYmC60iT+lRbVy26m4DR2F42Fk8x2BfWCiUYDYZ7Rj9W8=",
 "RjPTOAIdoC4KSh35fEv6CG5E2nZEKlqmUzeQYjykfE1ZVWpp/nQeLLC8tkEW/p/P5F2LuneTOVGVbpxhHxgJhZxorigAzis0dmGexoORqvgm74pbvFMHGzw+/FeFEGRp1zyEG6v+1UFG1ilG+TzWoPbECRSBPIHV5KBvzF3wsaA4YsTN5xZ9me5nR1h1DDqGaqIvhc82/KQizpB3fEKSvaNwCNeeP6yz3EXhieR5jwM6MeGL4Rg8mxKEtIroZJqpMxErYiRt4GalaoeEHoULeC94h+fwboyO5K1LCIjbEhG/uNV9YGsK9hFl/NxRSJyNUsDuIqEVLCRBnD4zcvCMl6UUX6VWNVHUP4FztO8ES6Kw3GZkQDSFnfcrkvJgio2F84UcxO7I8nP4BF5Tbqf+2ut6QGJPA/mHm06bBmChkA4CeMvUDEvTD/EL7vqtBBrzV+ZIy0B34ywIZV1zobH/aEuziCSlGjqXSqGI3aNXN7+7zOtbIwcCA/RgA21XhZbZWma3XMtMyyaZPi6pNu/YdDaHUqQFHKCR8G5RIgA/ZnjGxZgeqGGZPteUDugIoqqLZm3j1F+0kT76Q3d7H3jyA/kZ/ntG9/ZClgw6h2E/cJWWD+OkdYNT0s4tU9nZQwDe2ETX84Dy81kOvzUpMIQuVoFLOMlpLZFomN7FZ+5I3FlzcGqlXmBORF1E6zSLaDBnOKDzwrpUlvKNk5CZX1ZmPoeijUIFcavW4NhCO5FPvoA/ZWLC+9xgsinxo+L1O81EByza8KGZiA2u9uXSV8/wLyIPKSkORtEUWn2qcOe2sSKldICyQJcTncBjAOXbq/d7JKrG0PkSb/RZEbNX/GK1C0gIOe5ezNaan6zwB4zcQwz61fe2dogYkEpcDSvEgmMl74zENh9y8+7jUYFYaYC/fZepPd15Gyxglcg6mVLcAJwfqlUXiCutjZS3MVvSjssrN7YzXhV7IMvIwkbjkGzm9uuVqBw1f1nadJBvbhM/15z787R0mRmLMfHq3OHZwtOA2oO5zE3EgQFxx4jfcL47jYPnnBrAn+wQ5HmTQaqxZ+16t672JMDmfmiNolZPT2eLZRYKdfq+GOZXOqVeW4wu+ySnbJIxrqNcbpw2Jd4lF5dDOrexNmSbpA4iwspwK/qhdcO/w2+ySDOrtNdylz321P05um8U5TsTJs99w5Y+NMoywnhaxBg/qF05ZM80Dm9OeAF3BLEAZisS9dfAOB+f6kY4zwm2cUILeQQw8U/Djx7EzvkQOuq4Cx7fPAz7Lqx0XIJQGxtPhIZ3Hgv10a+4IMc0OSm01Q==",
 "aWtF84Xjj7nQK7+eegbVjmKFTUP6UM4wxyGav9kNel0rvJbQzy7n8te32gkRyl09oizQRQ9/VHx2ojJ3KP+q0GwVlcXj2pJ4GnbDH0yoENhYdjAOh3upxpKSrljdBYJ9HSt0QUbOWNKbm0xUek1etR2F89kRYho9GW4PS6QpspH2VJfDE+Cj7Hd5i30WWAjxCgYNkaBYyknfGvoeJgB2sqdUVzcTdt3hrIRddPfYZYkmuhhdATmTtpKdnYjphkBOgqBChWIDzXOCDVFpI4IyJYxuBM/c4MROsWxS4TVbvH/ZzZJxkOd0LUirK8t4H4CKnHxpQXUeUk2PGe3zPZL6uwy77FhzMdSxEI6+JaVTV/8GhRVMcj0GXsoRoR3JAMaT6xA+ysgCFSxfD7aBJ0URiqk6MDQb9sx/JtarV6x44K0d7fIsSrhtkHmtQ73oV9vRY8Z1SNySamXx0Y67xSa1WX7c2GlMsO4XDz02m3YEoCRUAwRA6CL/weUqF+9ny77O1j3UCI3BARcYKZIbEPpIh2kUYj08DH8fLBf3eEE40/c0KNjhi/0axtCllk0stkACbiKiQVAwU9L/JvHnb0Q6OWjdrIDZcIYKpLSpigVhLh+vn1B2jiiyR/JUU/u3gJyH8GsdUIhjbz7xJ8CFki7yHEv1f726MC8ufw4txOOxoGIgzQ0J4XYW32KnUEnYzk396psHqL9TGUPjuMvNHnOJICSYsqmTjD+zDx3otO4BHVm6m1Ttt9ccWI6Ja7JdA+DSLILljLKtw5CBG2nC0zuecyk7dmjCuJEGSbJSudj/bJhl3t5QCJ6Q/gf3IDEZ3gSLi9X0cGAt7ZiVfH6zUys1UyK7YW1o/MaO/VNBbZi2eruJqme/EYc4kMMzNdQ8M/1R52N3xwzqY4kexUd29aE64/G9DnGNtqHrQUsGSeJhMQjJ/qIORTp9H/ABNz4kvgJ8XzdzZglkAm01qbXDmsPW7CuHc0zElZnXo4pwkLm5zGNU84WoTJcPTEK/kOLHiInlMqJmoskahdnBemKnV/oiby/DyiXHvaHUBevqYmuC5nvkcC3Emyzu19ojMMLdActBaFLaqgm7Fl7feI+/CTxNx8PL9h3+yUVwmMtiH+BOKhgxo0OZBusIFfm+FK2vtQABM0RupiPuQlGFC51y2+eEdZBN3/jt8TJlHHnIsXexOtXQnu5clVTPUELcKhOHIJzfUG1cNU9MZrY85nxO/mjn22k4VsoAR+tF0uoOuIZhs/59iFkdUbVpjXUWe3ji75PSClq01i5Pm29g+PE6gaGdyvHtNPKSt7+n0Jwve7fpprAScg6L7+wYhAUAvfQ4YSw6yP8pE+/glEidcioK0MwV9YGtwuHlvthCqmfLbDhobyrmbK4SYVdbj8R25xlpD3eeDIvpO/PVb3ruPHh2MKbYckhWpQ5K3+Mwm/Kp3f2Nfz4LZ6iJIfX4k6FdVNl/4A1ERuEqY4z3XVHod/R91kJwALt54sdodpeTjVbqXyYzrCzx86TRMZ9WRXROfLFOeA=="
]

def _strumien(haslo, n):
    out, i = b"", 0
    while len(out) < n:
        out += hashlib.sha256(f"{haslo}:{i}".encode("utf-8")).digest(); i += 1
    return out[:n]

def odblokuj(haslo):
    h = unicodedata.normalize("NFKD", haslo).encode("ascii", "ignore").decode().lower().replace(" ", "")
    for nr, blob in enumerate(_HINTY, 1):
        dane = base64.b64decode(blob)
        try:
            tekst = zlib.decompress(bytes(a ^ b for a, b in zip(dane, _strumien(h, len(dane))))).decode("utf-8")
        except Exception:
            continue
        display(Markdown(tekst)); return
    print("Nieprawidłowe hasło.")

# odblokuj("...")
odblokuj('fortepian')
odblokuj('latawiec')


### Hint 1 (20. minuta) — filtr Frangiego, ROC

**Dane.** `X_te` to uint8 RGB `(20, 584, 565, 3)`; `Y_te` i `FOV_te` to bool `(20, 584, 565)`. Do filtru podaj tablicę 2-D float w [0, 1]:
kanał zielony `X[k][:, :, 1] / 255.0` (najlepszy kontrast naczyń) albo szarość — różnica w AUC jest minimalna (`frangi` i tak przelicza wejście na [0, 1]).

**Polaryzacja.** `frangi` ma parametr `black_ridges` (domyślnie `True`) — wykrywa struktury **ciemniejsze** od tła. Naczynia siatkówki są ciemne,
więc domyślna wartość jest właściwa. Jeśli odpowiedź wygląda jak „negatyw” (jasne tło między naczyniami, naczynia czarne), to ten parametr.

**Skale.** `sigmas` ≈ promień wykrywanych naczyń w pikselach. W DRIVE naczynia mają 1–12 px (mediana ~3), więc `np.arange(1, 4.01, 0.5)`
wystarcza. 19 skal z kodu wykładu (do σ = 10) liczy się ~3× dłużej, a w scikit-image 0.19 nawet obniża AUC (0,92 → 0,90). Czas: 0,3–1 s na obraz.

**Zakres wartości zależy od wersji scikit-image** (komórka 0 ją wypisuje): w 0.19 (środowisko Prof. Hachaja, domyślne `gamma=15`) odpowiedzi
są rzędu 10⁻⁸–10⁻⁴, od 0.20 (Colab, `gamma` dobierane automatycznie) — 10⁻⁴–0,9. Jedno i drugie jest normalne: to nie są prawdopodobieństwa,
więc do wyświetlania i progów używaj kwantyli, nie stałych (do *wyświetlenia*: podziel przez 99. percentyl i `np.clip(…, 0, 1)`).
**Nie zaokrąglaj odpowiedzi do uint8** (kod wykładu tak zapisuje pliki): kosztuje to 0,05–0,1 AUC, bo prawie cała informacja siedzi w ułamku zakresu.

**ROC.** `roc_curve(y, p)` i `roc_auc_score(y, p)` chcą wektorów 1-D. Dla tablic `(20, 584, 565)` indeksowanie maską `P[FOV_te]`
daje od razu wektor pikseli ze wszystkich 20 obrazów (ok. 4,5 mln); to samo dla `Y_te[FOV_te]`. Liczymy **zbiorczo** dla całego zbioru
testowego (jak `baseline_aug_predict.py` z wykładu), nie średnią z 20 krzywych. AUC wyraźnie poniżej 0,5 = odwrócona polaryzacja;
ok. 0,79–0,87 zamiast ok. 0,86–0,92 — najpewniej zaokrąglenie do uint8 albo ROC bez maski FOV.


### Hint 2 (40. minuta) — U-Net: architektura, łaty, konfiguracja treningu

**Kod wykładu (`src/baseline_aug.py`)**: `get_unet` można przepisać 1:1 (`activation=ReLU` to warstwa wywoływana `ReLU()(…)`;
`custom_activation`/`focal_loss`/`K` są nieużywane — pomiń). Generator `gen()` + `aug_utils` można zostawić (czyta obrazy z dysku
po ścieżkach), ale prościej jest użyć `losuj_laty` z komórki 0, która daje gotowe tablice. Alternatywnie napisz własną funkcję
z parametrem „baza filtrów”: wykład ma 32→64→128→256→512 (7,8 mln parametrów); baza 16 daje ~1,9 mln i trenuje ~3,5× szybciej,
a przy tej samej liczbie epok jest tylko nieznacznie gorsza (rzędu 0,01 AUC).

**Zamiast generatora `gen()`**: `laty, maski = losuj_laty(X_tr, Y_tr, ILE, 64, rng)` daje gotowe tablice float32 →
`model.fit(laty, maski, batch_size=32, epochs=…, validation_split=0.1)`. 4000 łat to ~200 MB — mieści się bez problemu.

**Ile?** Wykład: 100 epok × 2000 łat (200 tys.) — nie na zajęcia. Rozsądnie: 4000 łat, 12 epok. Czas na epokę przy bazie 16:
**CPU — ok. 40 s na 2 rdzeniach** (laptop z 4–8 rdzeniami: szacunkowo szybciej; baza 32: ~3,5× dłużej); GPU T4 — szacunkowo kilka sekund.
Po pierwszej epoce zobaczysz czas i policzysz, ile epok się zmieści. Żeby U-Net wyprzedził filtr Frangiego także w Dice, potrzeba ok. 10 epok
(baza 16, 4000 łat); po 3–5 epokach AUC jest już wyższe niż Frangiego, ale Dice bywa niższe — to nie błąd, tylko za krótki trening.
Jeśli nie zdążysz: tyle epok, ile się zmieści, **albo** wagi Prof. Hachaja (komórka 2d) — z komentarzem.

**Czy się uczy?** Strata z pierwszej epoki (średnia z epoki) wynosi ok. 0,35 — tuż przy „podłodze” ≈ 0,33–0,37, czyli entropii stałej
predykcji „tyle procent naczyń, ile jest w łatach” (`maski.mean()`); to normalne. W 2.–3. epoce powinna wyraźnie spaść (ok. 0,3 → 0,2).
Jeśli po 4–5 epokach nadal stoi przy podłodze, sieć przewiduje stałą — sprawdź, czy maski są float {0, 1} kształtu `(…, 1)` i czy wyjście
ma sigmoid. Dobry trening: poniżej 0,15 po ok. 6 epokach, AUC na łatach walidacyjnych > 0,95.

**Predykcja na całym obrazie.** Sieć ma wejście `(None, None, 3)`, więc przyjmuje 584×565 — ale 4 poolingi wymagają wymiarów podzielnych
przez 16, inaczej błąd przy `concatenate`. Stąd `dopelnij16(obraz / 255.0)` (→ 592×576, odbiciem), `model.predict(…[None], verbose=0)`,
wynik przyciąć `[0, :584, :565, 0]`. Kod wykładu zamiast tego skaluje obraz do 576×576 i z powrotem — działa, ale interpolacja rozmywa
najcieńsze naczynia. Obraz do sieci w [0, 1] jak łaty (uint8 bez dzielenia = śmieci na wyjściu).
